# 01. Training Experiments Report
This notebook is a **professional experimentation report** for transfer-learning-based pneumonia detection.
## Problem Statement
The objective is to classify chest X-rays into `NORMAL` vs `PNEUMONIA` while minimizing clinically critical errors, especially false negatives.
## Why Compare Multiple CNN Backbones?
Different architectures can produce different trade-offs between recall, specificity, robustness, and training cost.
## Role of Transfer Learning
Transfer learning accelerates convergence and improves generalization by reusing pretrained visual representations, which is essential for medical imaging projects with moderate dataset size.
## Notebook Scope
- Document experiments and assumptions.
- Compare model performance and error patterns.
- Justify the final model choice with evidence.
- Summarize experiment tracking through MLflow.
- Provide a deployment-oriented recommendation.

In [ ]:
# 02. Experiment Configuration - setup
from copy import deepcopy
from pathlib import Path
from pprint import pprint
import os
import sys
import time

import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc,
    classification_report,
 )
from IPython.display import display, Markdown

# Make src imports work when notebook is launched from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.data_loader import build_datasets, get_data_augmentation
from src.model import build_transfer_model

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

config = load_config(PROJECT_ROOT / "config.yaml")

# Resolve project-relative paths from config for notebook execution
for section, key in [("data", "data_dir"), ("training", "model_dir"), ("evaluation", "report_dir")]:
    p = Path(config[section][key])
    if not p.is_absolute():
        p = (PROJECT_ROOT / p).resolve()
    config[section][key] = str(p)

training_fig_dir = (PROJECT_ROOT / "reports" / "figures" / "training").resolve()
training_fig_dir.mkdir(parents=True, exist_ok=True)

def save_training_figure(name: str):
    """Save the current matplotlib figure into reports/figures/training."""
    out = training_fig_dir / name
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    return out

print("Project root:", PROJECT_ROOT)
print("Training figures directory:", training_fig_dir)
print("TensorFlow version:", tf.__version__)

In [ ]:
# 02. Experiment Configuration - summary table and interpretation
cfg = config
cfg_table = pd.DataFrame(
    [
        {"parameter": "backbone (default)", "value": cfg["model"]["backbone"]},
        {"parameter": "image_size", "value": tuple(cfg["data"]["img_size"])},
        {"parameter": "batch_size", "value": int(cfg["training"]["batch_size"])},
        {"parameter": "learning_rate", "value": float(cfg["training"]["learning_rate"])},
        {"parameter": "optimizer", "value": "Adam"},
        {"parameter": "loss", "value": "binary_crossentropy"},
        {"parameter": "epochs", "value": int(cfg["training"]["epochs"])},
        {"parameter": "seed", "value": int(cfg["general"]["seed"])},
        {"parameter": "trainable_layers", "value": int(cfg["model"]["trainable_layers"])},
        {"parameter": "callbacks", "value": "EarlyStopping, ReduceLROnPlateau, ModelCheckpoint"},
        {"parameter": "TensorFlow version", "value": tf.__version__},
    ]
 )

display(Markdown("## 02. Experiment Configuration"))
display(cfg_table)

display(Markdown("### Interpretation"))
display(Markdown(
    "Hyperparameters are set to prioritize stable transfer learning: moderate learning rate, controlled fine-tuning depth, and callback-based regularization. "
    "This configuration reduces overfitting risk while preserving enough capacity to adapt pretrained features to chest X-ray patterns."
))

In [ ]:
# 03. Dataset Validation
seed = int(config["general"]["seed"])
img_size = tuple(config["data"]["img_size"])
batch_size = int(config["training"]["batch_size"])

datasets = build_datasets(
    data_dir=config["data"]["data_dir"],
    img_size=img_size,
    batch_size=batch_size,
    seed=seed,
)

class_names = sorted([p.name for p in Path(config["data"]["data_dir"], "train").iterdir() if p.is_dir()])

validation_rows = []
for split_name in ("train", "val", "test"):
    ds = datasets[split_name]
    first_images, first_labels = next(iter(ds))
    total_batches = int(ds.cardinality().numpy()) if ds.cardinality().numpy() > 0 else -1
    validation_rows.append(
        {
            "split": split_name,
            "batch_shape_images": tuple(first_images.shape),
            "batch_shape_labels": tuple(first_labels.shape),
            "num_batches": total_batches,
        }
    )

validation_df = pd.DataFrame(validation_rows)

display(Markdown("## 03. Dataset Validation"))
display(validation_df)
print("Detected classes:", class_names)

images, labels = next(iter(datasets["train"]))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    if i >= len(images):
        ax.axis("off")
        continue
    img = tf.cast(images[i], tf.uint8).numpy()
    label_id = int(labels[i].numpy().item())
    label_id = max(0, min(label_id, len(class_names) - 1))
    ax.imshow(img)
    ax.set_title(class_names[label_id])
    ax.axis("off")
save_training_figure("dataset_validation_batch_preview.png")
plt.show()

display(Markdown("### Interpretation"))
display(Markdown(
    "Dataset validation confirms class discovery, batch tensor consistency, and end-to-end data pipeline readiness. "
    "This checkpoint is mandatory before training because data-shape or label issues can silently invalidate experiment conclusions."
))

In [ ]:
# 04. Model Architecture
def get_param_stats(model: tf.keras.Model):
    """Return total/trainable/non-trainable parameter counts."""
    total = int(model.count_params())
    trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
    non_trainable = total - trainable
    return total, trainable, non_trainable

architecture_rows = []
for backbone in ("vgg16", "resnet50v2"):
    m = build_transfer_model(
        backbone=backbone,
        input_shape=(img_size[0], img_size[1], 3),
        learning_rate=float(config["training"]["learning_rate"]),
        dropout_rate=float(config["model"]["dropout_rate"]),
        dense_units=int(config["model"]["dense_units"]),
        trainable_layers=int(config["model"]["trainable_layers"]),
        use_augmentation=bool(config["model"]["use_augmentation"]),
        augmentation_layer=get_data_augmentation(),
    )
    total, trainable, frozen = get_param_stats(m)
    architecture_rows.append(
        {
            "backbone": backbone,
            "total_params": total,
            "trainable_params": trainable,
            "frozen_params": frozen,
        }
    )

architecture_df = pd.DataFrame(architecture_rows)
display(Markdown("## 04. Model Architecture"))
display(architecture_df)

display(Markdown("### Interpretation"))
display(Markdown(
    "Transfer learning is used to reuse pretrained visual representations while fine-tuning only a subset of layers. "
    "This strategy reduces training time and data requirements, and usually improves generalization for medical imaging tasks."
))

In [ ]:
# 05-11. Experiment execution, evaluation, and comparison helpers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

def evaluate_predictions_for_report(y_true, y_prob, threshold=0.5):
    """Compute core classification metrics and confusion components."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / max((tn + fp), 1)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)

    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auc": float(roc_auc),
        "specificity": float(specificity),
        "sensitivity": float(recall),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
    }

def build_test_paths():
    """Build deterministic test file list in class order for error analysis."""
    test_root = Path(config["data"]["data_dir"]) / "test"
    paths = []
    for cls in class_names:
        class_paths = sorted([p for p in (test_root / cls).iterdir() if p.is_file()])
        paths.extend([str(p) for p in class_paths])
    return paths

def run_experiment_with_history(backbone, epochs_override=None):
    """Train a backbone and return history, metrics, and runtime."""
    start = time.perf_counter()
    lr = float(config["training"]["learning_rate"])
    epochs = int(epochs_override if epochs_override is not None else config["training"]["epochs"])

    model = build_transfer_model(
        backbone=backbone,
        input_shape=(img_size[0], img_size[1], 3),
        learning_rate=lr,
        dropout_rate=float(config["model"]["dropout_rate"]),
        dense_units=int(config["model"]["dense_units"]),
        trainable_layers=int(config["model"]["trainable_layers"]),
        use_augmentation=bool(config["model"]["use_augmentation"]),
        augmentation_layer=get_data_augmentation(),
    )

    model_output = Path(config["training"]["model_dir"]) / f"notebook_{backbone}.keras"

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=int(config["training"]["patience"]), restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7),
        ModelCheckpoint(model_output, monitor="val_loss", save_best_only=True),
    ]

    history_obj = model.fit(
        datasets["train"],
        validation_data=datasets["val"],
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )

    y_prob = model.predict(datasets["test"], verbose=0).ravel()
    y_true = np.concatenate([y.numpy().ravel() for _, y in datasets["test"]]).astype(int)
    metrics = evaluate_predictions_for_report(y_true, y_prob, threshold=float(config["evaluation"]["threshold"]))

    elapsed = time.perf_counter() - start
    return {
        "backbone": backbone,
        "model_path": str(model_output),
        "history": history_obj.history,
        "metrics": metrics,
        "train_time_sec": float(elapsed),
        "test_paths": build_test_paths(),
    }


def evaluate_saved_model(backbone, model_path):
    """Evaluate an existing saved model and return report-ready metrics."""
    start = time.perf_counter()
    model = tf.keras.models.load_model(model_path)
    y_prob = model.predict(datasets["test"], verbose=0).ravel()
    y_true = np.concatenate([y.numpy().ravel() for _, y in datasets["test"]]).astype(int)
    metrics = evaluate_predictions_for_report(y_true, y_prob, threshold=float(config["evaluation"]["threshold"]))
    elapsed = time.perf_counter() - start
    return {
        "backbone": backbone,
        "model_path": str(model_path),
        "history": {},
        "metrics": metrics,
        "train_time_sec": float("nan"),
        "inference_eval_time_sec": float(elapsed),
        "test_paths": build_test_paths(),
    }

RUN_EXPERIMENTS = False
EXPERIMENT_EPOCHS = 3
TARGET_BACKBONES = ("vgg16", "resnet50v2")

results_by_model = {}
if RUN_EXPERIMENTS:
    for b in TARGET_BACKBONES:
        print(f"Running experiment for {b}...")
        results_by_model[b] = run_experiment_with_history(b, epochs_override=EXPERIMENT_EPOCHS)
else:
    model_dir = Path(config["training"]["model_dir"])
    candidate_paths = {
        "vgg16": [model_dir / "best_vgg16.keras", model_dir / "notebook_vgg16.keras"],
        "resnet50v2": [model_dir / "best_resnet50v2.keras", model_dir / "notebook_resnet50v2.keras"],
    }
    # Also evaluate default model if present and backbone matches.
    default_model_path = model_dir / config["training"]["model_output_name"]
    if default_model_path.exists():
        default_backbone = str(config["model"]["backbone"]).lower()
        candidate_paths.setdefault(default_backbone, []).append(default_model_path)

    for b in TARGET_BACKBONES:
        found = next((p for p in candidate_paths.get(b, []) if p.exists()), None)
        if found is not None:
            print(f"Evaluating saved model for {b}: {found.name}")
            results_by_model[b] = evaluate_saved_model(b, found)

print("Available experiment results:", list(results_by_model.keys()))

In [ ]:
# 05. Training Curves
display(Markdown("## 05. Training Curves"))

def plot_training_curves(results):
    """Plot train/val loss, accuracy, and AUC for all available model histories."""
    if not results:
        print("No model results available.")
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    plotted = False

    for model_name, result in results.items():
        history = result.get("history", {})
        if not history:
            continue
        plotted = True
        axes[0].plot(history.get("loss", []), label=f"{model_name} - train")
        axes[0].plot(history.get("val_loss", []), linestyle="--", label=f"{model_name} - val")

        axes[1].plot(history.get("accuracy", []), label=f"{model_name} - train")
        axes[1].plot(history.get("val_accuracy", []), linestyle="--", label=f"{model_name} - val")

        axes[2].plot(history.get("auc", []), label=f"{model_name} - train")
        axes[2].plot(history.get("val_auc", []), linestyle="--", label=f"{model_name} - val")

    axes[0].set_title("Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[1].set_title("Accuracy Curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[2].set_title("AUC Curves")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("AUC")

    if plotted:
        for ax in axes:
            ax.legend(fontsize=8)
        save_training_figure("training_curves_comparison.png")
        plt.show()
    else:
        plt.close(fig)
        print("No per-epoch histories available. Enable RUN_EXPERIMENTS=True to generate training curves.")

plot_training_curves(results_by_model)

display(Markdown("### Interpretation"))
display(Markdown(
    "When available, training/validation curves are used to assess convergence, stability, and overfitting. "
    "A small gap between train and validation trends generally indicates stronger generalization behavior."
))

In [ ]:
# 06. Evaluation Metrics
display(Markdown("## 06. Evaluation Metrics"))

metric_rows = []
for model_name, result in results_by_model.items():
    m = result["metrics"]
    metric_rows.append(
        {
            "model": model_name,
            "accuracy": round(m["accuracy"], 4),
            "precision": round(m["precision"], 4),
            "recall": round(m["recall"], 4),
            "f1": round(m["f1"], 4),
            "roc_auc": round(m["auc"], 4),
            "specificity": round(m["specificity"], 4),
            "sensitivity": round(m["sensitivity"], 4),
        }
    )

metrics_df = pd.DataFrame(metric_rows).sort_values("roc_auc", ascending=False) if metric_rows else pd.DataFrame()
display(metrics_df)

display(Markdown("### Interpretation"))
display(Markdown(
    "In medical screening, **recall/sensitivity** is critical to reduce missed pneumonia cases (false negatives), while **specificity** controls false alarms. "
    "Accuracy alone is insufficient under class imbalance and must be interpreted alongside AUC, precision, and recall."
))

## 07. Confusion Matrix
This section quantifies true positives, true negatives, false positives, and false negatives for the selected candidate model.

In [ ]:
# 07. Confusion Matrix
if not results_by_model:
    print("No available results to plot confusion matrix.")
else:
    best_model_name = max(results_by_model, key=lambda k: results_by_model[k]["metrics"]["auc"])
    m = results_by_model[best_model_name]["metrics"]
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"Confusion Matrix - {best_model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticklabels(["NORMAL", "PNEUMONIA"])
    ax.set_yticklabels(["NORMAL", "PNEUMONIA"], rotation=0)
    save_training_figure("confusion_matrix_best_model.png")
    plt.show()

    display(pd.DataFrame([{"TP": m["tp"], "TN": m["tn"], "FP": m["fp"], "FN": m["fn"]}]))

display(Markdown("### Interpretation"))
display(Markdown(
    "False negatives are clinically costly because pneumonia cases are missed. "
    "Model selection should prioritize high recall while maintaining acceptable specificity for operational workload."
))

## 08. ROC Curve
ROC analysis compares discrimination performance across thresholds and across candidate models.

In [ ]:
# 08. ROC Curve
if not results_by_model:
    print("No available results to plot ROC curve.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    for model_name, result in results_by_model.items():
        y_true = result["metrics"]["y_true"]
        y_prob = result["metrics"]["y_prob"]
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_title("ROC Curve Comparison")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()
    save_training_figure("roc_curve_comparison.png")
    plt.show()

display(Markdown("### Interpretation"))
display(Markdown(
    "AUC captures threshold-independent separability. Higher AUC indicates stronger discrimination between NORMAL and PNEUMONIA across operating points."
))

## 09. Precision-Recall Curve
Precision-Recall analysis is emphasized because class imbalance can hide weaknesses when using ROC or accuracy alone.

In [ ]:
# 09. Precision-Recall Curve
if not results_by_model:
    print("No available results to plot Precision-Recall curve.")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    for model_name, result in results_by_model.items():
        y_true = result["metrics"]["y_true"]
        y_prob = result["metrics"]["y_prob"]
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        pr_auc = auc(rec, prec)
        ax.plot(rec, prec, label=f"{model_name} (PR AUC={pr_auc:.3f})")
    ax.set_title("Precision-Recall Curve Comparison")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend()
    save_training_figure("pr_curve_comparison.png")
    plt.show()

display(Markdown("### Interpretation"))
display(Markdown(
    "Precision-Recall curves are especially informative under class imbalance. "
    "They directly show the trade-off between identifying pneumonia cases (recall) and limiting false alarms (precision)."
))

## 10. Error Analysis
False positives and false negatives are reviewed to identify recurring failure patterns and possible data-driven causes.

In [ ]:
# 10. Error Analysis
if not results_by_model:
    print("No available results for error analysis.")
else:
    best_model_name = max(results_by_model, key=lambda k: results_by_model[k]["metrics"]["auc"])
    result = results_by_model[best_model_name]
    m = result["metrics"]

    y_true = m["y_true"]
    y_pred = m["y_pred"]
    test_paths = result.get("test_paths", [])

    fp_idx = [i for i, (yt, yp) in enumerate(zip(y_true, y_pred)) if yt == 0 and yp == 1][:6]
    fn_idx = [i for i, (yt, yp) in enumerate(zip(y_true, y_pred)) if yt == 1 and yp == 0][:6]

    print(f"Best model for analysis: {best_model_name}")
    print(f"False positives sampled: {len(fp_idx)}")
    print(f"False negatives sampled: {len(fn_idx)}")

    def _show_error_grid(indices, title):
        if not indices:
            print(f"No samples for {title}")
            return
        fig, axes = plt.subplots(2, 3, figsize=(11, 6))
        for ax, idx in zip(axes.ravel(), indices):
            p = Path(test_paths[idx])
            img = plt.imread(p)
            ax.imshow(img, cmap="gray")
            ax.set_title(p.name, fontsize=8)
            ax.axis("off")
        for ax in axes.ravel()[len(indices):]:
            ax.axis("off")
        fig.suptitle(title)
        save_training_figure(title.lower().replace(" ", "_") + ".png")
        plt.show()

    _show_error_grid(fp_idx, "False Positive Examples")
    _show_error_grid(fn_idx, "False Negative Examples")

display(Markdown("### Interpretation"))
display(Markdown(
    "Error analysis helps identify systematic failure modes (e.g., low contrast, atypical opacity patterns, acquisition artifacts). "
    "These insights guide next-step actions such as stronger preprocessing, targeted augmentation, and threshold tuning."
))

## 11. Experiment Comparison
Candidate backbones are compared on KPI metrics and runtime to support an evidence-based final selection.

In [ ]:
# 11. Experiment Comparison
if metrics_df.empty:
    print("No metrics available for comparison.")
else:
    comparison_df = metrics_df.copy()
    time_rows = []
    for model_name, result in results_by_model.items():
        time_rows.append({"model": model_name, "training_time_sec": result.get("train_time_sec", np.nan)})
    time_df = pd.DataFrame(time_rows)
    comparison_df = comparison_df.merge(time_df, on="model", how="left")
    display(comparison_df)

    # Metrics comparison chart
    plot_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "specificity"]
    melted = comparison_df.melt(id_vars=["model"], value_vars=plot_metrics, var_name="metric", value_name="value")

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    sns.barplot(data=melted, x="metric", y="value", hue="model", ax=axes[0])
    axes[0].set_title("Metric Comparison Across Models")
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=30)

    sns.barplot(data=comparison_df, x="model", y="training_time_sec", ax=axes[1])
    axes[1].set_title("Training Time Comparison (sec)")
    axes[1].set_ylabel("seconds")

    save_training_figure("model_comparison_metrics_and_time.png")
    plt.show()

display(Markdown("### Interpretation"))
display(Markdown(
    "Model selection should balance clinical priorities (high recall/sensitivity), discrimination strength (AUC), and operational cost (training/runtime). "
    "The preferred backbone is the one that offers the best clinical-risk trade-off under reproducible conditions."
))

## 12. MLflow Summary
This section reports experiment tracking context and run-level metadata to support reproducibility and auditability.

In [ ]:
# 12. MLflow Summary
mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
exp_name = config["mlflow"]["experiment_name"]
exp = mlflow.get_experiment_by_name(exp_name)

display(Markdown(f"**Experiment name:** `{exp_name}`"))
if exp is None:
    print("No MLflow experiment found for this name yet.")
else:
    print("Experiment ID:", exp.experiment_id)
    runs_df = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        max_results=10,
        order_by=["attributes.start_time DESC"],
    )
    show_cols = [c for c in runs_df.columns if c.startswith("params.") or c.startswith("metrics.")]
    show_cols = ["run_id", "status", "start_time"] + show_cols[:10]
    display(runs_df[show_cols])

display(Markdown("### Interpretation"))
display(Markdown(
    "MLflow enables reproducibility through centralized logging of parameters, metrics, and artifacts. "
    "It is essential for experiment governance, model comparison, and defensible model selection."
))

## 13. Conclusion
Final model recommendation is based on a combined view of clinical KPIs, discrimination quality, and operational constraints.

In [ ]:
# 13. Conclusion
if metrics_df.empty:
    display(Markdown(
        "No completed model result is currently available. Set `RUN_EXPERIMENTS=True` or provide saved models to generate a data-driven final recommendation."
    ))
else:
    best_row = metrics_df.sort_values(["roc_auc", "recall", "specificity"], ascending=False).iloc[0]
    best_model = best_row["model"]

    conclusion_text = f"""
The comparative analysis indicates that **{best_model}** currently provides the strongest overall trade-off between discrimination quality and clinical reliability.

Key evidence: ROC AUC = **{best_row['roc_auc']:.4f}**, Recall/Sensitivity = **{best_row['recall']:.4f}**, Specificity = **{best_row['specificity']:.4f}**, F1-score = **{best_row['f1']:.4f}**.

Given the clinical objective of reducing missed pneumonia cases, recall remains the primary decision metric, while specificity controls false-alert burden.

Based on these results, **{best_model} is recommended as the primary backbone for downstream deployment validation**, subject to external validation and threshold calibration.
"""
    display(Markdown(conclusion_text))